# 02 — Data Cleaning & Preparation

Goal: turn the raw taxi data into a reliable dataset for analysis.

Issues found in `01_initial_exploration`:
- `pickup` / `dropoff` stored as text, not datetime
- Missing values in `payment`, zone and borough columns
- Trips with 0 distance, 0 passengers, or drop-off at/before pickup
- Possible fare and speed outliers

**Principle:** fix what can be fixed, flag what is uncertain, and remove only records that are physically impossible.

In [1]:
# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
# ==========================================
# 2. LOAD RAW DATA
# ==========================================

df = pd.read_csv("../data/raw/taxis.csv", parse_dates=["pickup", "dropoff"])

raw_rows = len(df)
print(f"Raw shape: {df.shape}")
df.dtypes

Raw shape: (6433, 14)


pickup             datetime64[us]
dropoff            datetime64[us]
passengers                  int64
distance                  float64
fare                      float64
tip                       float64
tolls                     float64
total                     float64
color                         str
payment                       str
pickup_zone                   str
dropoff_zone                  str
pickup_borough                str
dropoff_borough               str
dtype: object

## Step 1 — Fix data types
`parse_dates` above converts `pickup` and `dropoff` to datetime. Text columns with a small, fixed set of values become `category`, which saves memory and makes grouping clearer.

In [3]:
# ==========================================
# 3. FIX DATA TYPES
# ==========================================

categorical_cols = ["color", "payment", "pickup_zone", "dropoff_zone",
                    "pickup_borough", "dropoff_borough"]

# Standardise text before converting: trim spaces, consistent casing
for col in categorical_cols:
    df[col] = df[col].str.strip()

df[["pickup", "dropoff"]].dtypes

pickup     datetime64[us]
dropoff    datetime64[us]
dtype: object

## Step 2 — Handle missing values
These columns are descriptive labels, not measurements. Dropping the rows would discard valid fare data, so missing values are filled with an explicit `"Unknown"` label instead.

In [4]:
# ==========================================
# 4. HANDLE MISSING VALUES
# ==========================================

print("Before:")
print(df.isna().sum()[df.isna().sum() > 0])

df["payment"] = df["payment"].fillna("unknown")
for col in ["pickup_zone", "dropoff_zone", "pickup_borough", "dropoff_borough"]:
    df[col] = df[col].fillna("Unknown")

df[categorical_cols] = df[categorical_cols].astype("category")

print("\nAfter:")
print(f"Total missing values: {df.isna().sum().sum()}")

Before:
payment            44
pickup_zone        26
dropoff_zone       45
pickup_borough     26
dropoff_borough    45
dtype: int64

After:
Total missing values: 0


## Step 3 — Derive trip duration and speed
These are needed to judge whether a trip is physically possible.

In [5]:
# ==========================================
# 5. DERIVED TRIP METRICS
# ==========================================

df["duration_min"] = (df["dropoff"] - df["pickup"]).dt.total_seconds() / 60
df["speed_mph"] = df["distance"] / (df["duration_min"] / 60)
df.loc[df["duration_min"] <= 0, "speed_mph"] = np.nan  # avoid inf / divide-by-zero

df[["duration_min", "speed_mph"]].describe().round(2)

,duration_min,speed_mph
count,6433.00,6427.00
mean,14.35,13.09
std,11.64,68.04
min,0.00,0.00
25%,6.50,7.57
50%,10.90,10.04
75%,18.52,13.71
max,107.67,4834.29


## Step 4 — Remove invalid trips
A trip is removed only if it is **impossible**:
| Rule | Why |
|---|---|
| `duration_min <= 0` | Drop-off cannot happen at or before pickup |
| `distance == 0` | No trip took place (cancelled / GPS error), yet fare was charged |
| `speed_mph > 60` | Not realistic for NYC city driving; indicates bad timestamps or distance |

In [6]:
# ==========================================
# 6. REMOVE INVALID TRIPS
# ==========================================

invalid_rules = {
    "Non-positive duration": df["duration_min"] <= 0,
    "Zero distance": df["distance"] == 0,
    "Speed > 60 mph": df["speed_mph"] > 60,
}

for name, mask in invalid_rules.items():
    print(f"{name}: {mask.sum()} rows")

invalid_mask = np.logical_or.reduce(list(invalid_rules.values()))
print(f"\nTotal rows removed (overlaps counted once): {invalid_mask.sum()}")

df = df[~invalid_mask].copy()

Non-positive duration: 6 rows
Zero distance: 51 rows
Speed > 60 mph: 11 rows

Total rows removed (overlaps counted once): 62


## Step 5 — Fix questionable values
`passengers == 0` is almost certainly a driver data-entry error. The trip itself is real (distance, fare, time all valid), so the row is kept and the passenger count is marked as missing instead of zero this keeps averages from being pulled down.

In [7]:
# ==========================================
# 7. FIX QUESTIONABLE VALUES
# ==========================================

print(f"Trips with 0 passengers: {(df['passengers'] == 0).sum()}")

df["passengers"] = df["passengers"].astype("Int64")  # nullable integer
df.loc[df["passengers"] == 0, "passengers"] = pd.NA

df["passengers"].value_counts(dropna=False).sort_index()

Trips with 0 passengers: 94


passengers
1       4625
2        871
3        242
4        110
5        277
6        152
<NA>      94
Name: count, dtype: Int64

## Step 6 — Flag outliers 
High fares can be legitimate. Instead of removing them, add a flag using the IQR rule so analyses can include or exclude them deliberately.

In [ ]:
# ==========================================
# 8. FLAG OUTLIERS
# ==========================================

q1, q3 = df["fare"].quantile([0.25, 0.75])
upper_fence = q3 + 1.5 * (q3 - q1)

df["is_fare_outlier"] = df["fare"] > upper_fence

print(f"Fare upper fence: ${upper_fence:.2f}")
print(f"Flagged as outliers: {df['is_fare_outlier'].sum()} ({df['is_fare_outlier'].mean():.1%})")

df.loc[df["is_fare_outlier"], ["distance", "duration_min", "fare", "total",
                               "pickup_zone", "dropoff_zone"]].nlargest(5, "fare")

Fare upper fence: $27.75
Flagged as outliers: 577 (9.1%)


,distance,duration_min,fare,total,pickup_zone,dropoff_zone
5364,36.70,64.850000,150.0,174.82,JFK Airport,JFK Airport
5648,33.46,89.883333,150.0,169.70,East Flushing,Unknown
4050,33.76,55.950000,143.5,144.30,LaGuardia Airport,Unknown
2231,26.35,46.400000,130.0,136.56,East Harlem North,Unknown
4815,10.36,25.366667,100.0,123.36,TriBeCa/Civic Center,Unknown


## Step 7 — Feature engineering
Add columns that will be useful for time-based and tipping analysis.

In [9]:
# ==========================================
# 9. FEATURE ENGINEERING
# ==========================================

df["pickup_date"] = df["pickup"].dt.date
df["pickup_hour"] = df["pickup"].dt.hour
df["pickup_day"] = pd.Categorical(
    df["pickup"].dt.day_name(),
    categories=["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"],
    ordered=True,
)
df["is_weekend"] = df["pickup"].dt.dayofweek >= 5

# Tip % only meaningful for card payments (cash tips are not recorded)
df["tip_pct"] = np.where(df["payment"] == "credit card", df["tip"] / df["fare"] * 100, np.nan)

df[["pickup_hour", "pickup_day", "is_weekend", "duration_min", "speed_mph", "tip_pct"]].head()

/var/folders/76/qlp_kr813x7fbxq00qyd32nr0000gn/T/ipykernel_1300/658056521.py:12: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["is_weekend"] = df["pickup"].dt.dayofweek >= 5


,pickup_hour,pickup_day,is_weekend,duration_min,speed_mph,tip_pct
0,20,Saturday,True,6.250000,15.360000,30.714286
1,16,Monday,False,7.083333,6.691765,NaN
2,17,Wednesday,False,7.400000,11.108108,31.466667
3,1,Sunday,True,25.866667,17.860825,22.777778
4,13,Saturday,True,9.533333,13.594406,12.222222


## Step 8 — Validate the cleaned data
Assertions fail loudly if any cleaning rule did not hold.

In [10]:
# ==========================================
# 10. VALIDATION CHECKS
# ==========================================

assert df["duration_min"].gt(0).all(), "Non-positive durations remain"
assert df["distance"].gt(0).all(), "Zero distances remain"
assert df["speed_mph"].le(60).all(), "Unrealistic speeds remain"
assert df["fare"].gt(0).all(), "Non-positive fares remain"
assert not df.duplicated().any(), "Duplicate rows found"
assert df[categorical_cols].notna().all().all(), "Missing categorical values remain"

print("All validation checks passed ✅")

All validation checks passed ✅


In [11]:
# ==========================================
# 11. CLEANING SUMMARY
# ==========================================

print(f"Raw rows:     {raw_rows}")
print(f"Clean rows:   {len(df)}")
print(f"Removed:      {raw_rows - len(df)} ({(raw_rows - len(df)) / raw_rows:.2%})")
print(f"Columns:      {df.shape[1]}")

df.info()

Raw rows:     6433
Clean rows:   6371
Removed:      62 (0.96%)
Columns:      22
<class 'pandas.DataFrame'>
Index: 6371 entries, 0 to 6432
Data columns (total 22 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   pickup           6371 non-null   datetime64[us]
 1   dropoff          6371 non-null   datetime64[us]
 2   passengers       6277 non-null   Int64         
 3   distance         6371 non-null   float64       
 4   fare             6371 non-null   float64       
 5   tip              6371 non-null   float64       
 6   tolls            6371 non-null   float64       
 7   total            6371 non-null   float64       
 8   color            6371 non-null   category      
 9   payment          6371 non-null   category      
 10  pickup_zone      6371 non-null   category      
 11  dropoff_zone     6371 non-null   category      
 12  pickup_borough   6371 non-null   category      
 13  dropoff_borough  6371 non-nul

In [12]:
# ==========================================
# 12. SAVE CLEANED DATA
# ==========================================

import os
os.makedirs("../data/processed", exist_ok=True)

df.to_csv("../data/processed/taxis_clean.csv", index=False)
print("Saved to ../data/processed/taxis_clean.csv")

Saved to ../data/processed/taxis_clean.csv


## Summary of cleaning decisions

| Issue | Action | Reason |
|---|---|---|
| `pickup`/`dropoff` stored as text | Parsed as datetime | Enables duration & time features |
| Missing `payment` | Filled with `"unknown"` | Keeps valid fare data |
| Missing zones/boroughs | Filled with `"Unknown"` | Keeps valid fare data |
| Duration ≤ 0 | **Removed** | Physically impossible |
| Distance = 0 | **Removed** | No trip occurred |
| Speed > 60 mph | **Removed** | Unrealistic in NYC traffic |
| Passengers = 0 | Set to missing | Data-entry error; trip still valid |
| High fares | **Flagged** (`is_fare_outlier`) | May be legitimate (airport trips) |
| — | Added `duration_min`, `speed_mph`, `pickup_hour`, `pickup_day`, `is_weekend`, `tip_pct` | Ready for analysis |

**Note:** when reloading the CSV, pass `parse_dates=["pickup", "dropoff"]` — CSV does not preserve dtypes.